A production-ready Geospatial Lakehouse that ingests streaming GPS events, enriches them with lat/lon grid indexes, and publishes Gold Delta tables for analytics & ML workloads.

You’ll learn how to:

- Build a streaming ETL pipeline using Delta Live Tables (DLT)
- Transform raw streams into clean GEOMETRY-based tables
- Create lat/lon grid index columns for fast spatial joins
- Publish Gold aggregates for BI dashboards, maps, and ML models
- Tune performance using Photon + Delta optimizations
- Enable ML workflows with spatial feature engineering + MLflow

[ Streaming GPS Events ] 

      ├─► BRONZE (raw delta)

      ├─► SILVER (cleaned + GEOMETRY + grid index)

      └─► GOLD (aggregations, joins, KPIs)

                   ├─ Dashboards (maps)
                   
                   └─ ML pipelines (spatial models)

### 1. Environment Setup

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS geo_demo;
USE CATALOG geo_demo;
CREATE SCHEMA IF NOT EXISTS mobility;
USE SCHEMA mobility;

### 2. BRONZE – Raw GPS Event Ingestion

In [0]:
from pyspark.sql.functions import col, expr, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
import uuid, time, random

# NYC base location
base_lat, base_lon = 40.75, -73.99

def make_rows(n=200, jitter=0.02):
    rows = []
    now = int(time.time())
    for i in range(n):
        rows.append((
            str(uuid.uuid4()),              # event_id
            f"device_{i%5}",                # device_id
            now,                            # unix timestamp
            base_lat + (random.random()-0.5)*jitter*2,   # lat jitter
            base_lon + (random.random()-0.5)*jitter*2    # lon jitter
        ))
    return rows

schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("device_id", StringType(), True),
    StructField("ts_unix",  DoubleType(), True),
    StructField("lat",      DoubleType(), True),
    StructField("lon",      DoubleType(), True),
])

df_bronze = spark.createDataFrame(make_rows(), schema) \
    .withColumn("ts", expr("timestamp_millis(CAST(ts_unix * 1000 AS BIGINT))")) \
    .withColumn("ingest_ts", current_timestamp()) \
    .withColumn("src_file", expr("'batch_1'")) \
    .drop("ts_unix")

df_bronze.show(5, truncate=False)

df_bronze.write.format("delta").mode("overwrite").saveAsTable("bronze_gps_raw")


### 3. SILVER – Clean + Enrich + Add Grid Indexing

In [0]:
%sql
CREATE OR REPLACE TABLE silver_gps_clean AS
SELECT
  event_id,
  device_id,
  ts,
  CASE WHEN lat BETWEEN -90 AND 90 THEN lat END AS lat,
  CASE WHEN lon BETWEEN -180 AND 180 THEN lon END AS lon,
  ST_POINT(lon, lat) AS geo_point,
  CONCAT(FLOOR(lat * 100), '_', FLOOR(lon * 100)) AS grid_100m,  -- 100m grid cell
  CAST(ts AS DATE) AS day,
  lat IS NOT NULL AND lon IS NOT NULL AS valid,
  ingest_ts,
  src_file
FROM bronze_gps_raw;

Adds a 100m² grid index (grid_100m) using FLOOR(lat * 100).

Adds geo_point for spatial functions (buffers, distance)

### 4. GOLD – Time + Grid Aggregations

In [0]:
%sql
CREATE OR REPLACE TABLE gold_mobility_grid_5m AS
SELECT
  grid_100m,
  WINDOW(ts, '5 minutes') AS w,
  COUNT(*) AS cnt_points,
  COUNT(DISTINCT device_id) AS cnt_devices
FROM silver_gps_clean
WHERE valid = TRUE
GROUP BY grid_100m, WINDOW(ts, '5 minutes');

In [0]:
%sql

select * from gold_mobility_grid_5m

This creates grid-level metrics for BI maps, anomaly detection, and ML.

In [0]:
from pyspark.sql.functions import expr

stores = [
    ("store_Apple", "Midtown", 40.7549, -73.9840),
    ("store_Samsung", "Chelsea", 40.7465, -74.0014),
]

df_stores = spark.createDataFrame(stores, ["store_id","name","lat","lon"]) \
    .withColumn("geo_point", expr("ST_POINT(lon, lat)")) \
    .withColumn("grid_100m", expr("concat(floor(lat * 100), '_', floor(lon * 100))"))
df_stores.write.mode("overwrite").saveAsTable("dim_stores")

In [0]:
%sql
WITH approx AS (
  SELECT s.store_id, s.geo_point AS store_loc, g.*
  FROM dim_stores s
  JOIN silver_gps_clean g
    ON s.grid_100m = g.grid_100m
  WHERE g.valid = TRUE
)
SELECT
  store_id,
  COUNT(*) AS cnt_visits,
  AVG(ST_DISTANCE(store_loc, geo_point)) AS avg_distance_m
FROM approx
WHERE ST_DISTANCE(store_loc, geo_point) <= 300  -- 300m radius
GROUP BY store_id;

### Live Version 

In [0]:
%sql
-- BRONZE Stream
CREATE OR REFRESH STREAMING LIVE TABLE bronze_gps_raw
AS SELECT
  *,
  current_timestamp() AS ingest_ts,
  input_file_name() AS src_file
FROM STREAM READ_FILES('/Volumes/geo_demo/mobility/incoming', format => 'json');

-- SILVER
CREATE OR REFRESH LIVE TABLE silver_gps_clean
AS SELECT
  *,
  ST_POINT(lon, lat) AS geo_point,
  CONCAT(FLOOR(lat*100), '_', FLOOR(lon*100)) AS grid_100m
FROM LIVE.bronze_gps_raw;

-- GOLD
CREATE OR REFRESH LIVE TABLE gold_mobility_grid_5m
AS SELECT
  grid_100m,
  WINDOW(ts, '5 minutes') AS w,
  COUNT(*) AS cnt_points,
  COUNT(DISTINCT device_id) AS cnt_devices
FROM STREAM(LIVE.silver_gps_clean)
GROUP BY grid_100m, WINDOW(ts, '5 minutes');

### 7. Spatial Feature Engineering for ML

In [0]:
from pyspark.sql.functions import expr, col
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
import mlflow

hex_data = spark.table("gold_mobility_grid_5m") \
    .withColumn("label", expr("cnt_devices > 10"))

feat = hex_data.select(
    col("cnt_points").cast("double"),
    col("cnt_devices").cast("double"),
    col("label").cast("double")
)

vec = VectorAssembler(
    inputCols=["cnt_points", "cnt_devices"],
    outputCol="features"
)
lr = LogisticRegression(
    featuresCol="features",
    labelCol="label"
)

with mlflow.start_run():
    model = lr.fit(vec.transform(feat))
    mlflow.spark.log_model(
        model,
        "model",
        dfs_tmpdir="/Volumes/geo_demo/mobility/geo_model"
    )
